# Day 3: Optimizing the Use of Finite Difference Methods

#### &#9989; **Write your name here**

Last class, we used FDMs and heat equation,

$$\frac{\partial^2 y_i}{\partial x^2} = \frac{y_{i-1}-2y_i+y_{i+1}}{h^2} \text{ }\text{ }\text{ }\text{ }\text{ }\text{ and }\text{ }\text{ }\text{ }\text{ }\text{ } \frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}$$

to demonstrate how to compute $\partial T_i / \partial t$ (`rate[i]` in the code) when implementing FDMs computationally:

$$\frac{\partial T_i}{\partial t} = \alpha \frac{(T_{i-1} - 2T_i + T_{i+1})}{\text{d}x^2}$$

With this setup in mind, consider a new set of initial conditions for our hot wire:

In [1]:
import numpy as np

N = 500

# wire starts at 100 degrees C, with ends held at 0 degrees C
T0 = 100 * np.ones(N)
T0[0] = 0
T0[-1] = 0

# wire is 10 cm long
x = np.linspace(0, 10, N)
dx = x[1] - x[0]

&#9989; **Task 0.1:** Create a clearly labeled plot that shows the initial temperature distribution of the wire.

In [2]:
# your answer here

&#9989; **Task 0.2:** The functions below are example solutions to Tasks 1.2 and 1.3 from the Day 2 assignment. **Add documention** to explain the purpose, inputs, and output of each function, **or just replace them** with your own functions and documentation from Day 2.

In [4]:
# add documentation or replace this with your own function from Day 2
def d2Tdx2(T, i, dx):
    return (T[i-1] - 2 * T[i] + T[i+1]) / dx**2

# add documentation or replace this with your own function from Day 2
def dTdt(T, N, dx, alpha=1.27e-4):
    rate = np.zeros(N)
    for i in range(1, N-1):
        rate[i] = alpha * d2Tdx2(T, i, dx)
    return rate

---
## Part 1: Time evolution of the temperature distribution

&#9989; **Task 1.1:** Implement code to update the temperature of the wire using FDMs, and create a plot that shows the temperature distribution at different snapshots in time: 
- At t = 0
- At 30 minutes
- At 4 hours
- At 12 hours
- At 1 full day

*Hint: Convert all time markers to seconds, and use `dt = 1`.*

In [5]:
# your answer here

After 1 full day, the temperature of the wire should be sitting just above 40 °C at the middle point. Our model assumes temperature can only leave the wire through the very ends of the wire, and can't dissipate through the surface, which is why it takes so long to cool down. It's as if the wire were suspended in a vacuum.

Suppose we wanted to see how long it takes the wire to fall below 1 °C. The code below could do it, but be warned, it takes a little while to run. The `time` module has been imported and used below to show just how long it takes.

In [7]:
import time

t = 0
dt = 1

T = np.copy(T0)

start = time.time()

while np.max(T) > 1:
    dT = dTdt(T, N, dx) * dt
    T += dT
    t += dt

end = time.time()

print(t/86400, "days to cool below 1 °C")
print(end - start, "seconds of run time")

4.475381944444444 days to cool below 1 °C
71.18607306480408 seconds of run time


See the slides for a breakdown of what could be taking so long here.

&#9989; **Task 1.2:** What is taking so long? Describe what you could change about the code or the functions from earlier to speed up the run time.

**/your answer here/**

#### &#128721; **Stop here and check in with an instructor.**

&#9989; **Task 1.3:** Make your proposed changes. Once you have your code working, calculate the wire temperature after 10 full days, and **print the overall run-time**.

In [ ]:
# your answer here

---
## Part 2: Divergent behavior in FDMs

Streamlining the calculation of $\partial T/\partial t$ is not the way to speed up the code. Another approach would be to reduce the number of iterations overall by increasing the time step. Bigger time steps means fewer steps to get to the final time.

However, when we try this approach, we start to run into issues and incorrect answers. See below.

In [11]:
t = 0

# increased time step from 1 to 5
dt = 5

T = np.copy(T0)

start = time.time()

while np.max(T) > 1:
    dT = dTdt(T, N, dx) * dt
    T += dT
    t += dt

end = time.time()

print(t/86400, "days to cool below 1 °C")
print(end - start, "seconds of run time")

0.024479166666666666 days to cool below 1 °C
0.08099913597106934 seconds of run time


/var/folders/w5/xpqhm5816ql9jlb8gyvxc2980000gn/T/ipykernel_90621/2906057025.py:3: RuntimeWarning: overflow encountered in scalar divide
  return (T[i-1] - 2 * T[i] + T[i+1]) / dx**2
/var/folders/w5/xpqhm5816ql9jlb8gyvxc2980000gn/T/ipykernel_90621/2626957295.py:12: RuntimeWarning: invalid value encountered in add
  T += dT


See the slides for deeper look into what's happening here.

&#9989; **Task 2.1:** What is the "overflow" that the warning message refers to? Why does the computer display values for the number of the days and the runtime that are clearly incorrect?

**/your answer here/**

&#9989; **Task 2.2:** Why does this divergent behavior happen when the time-step in increased? Map out an explanation with your peers on a whiteboard, and take notes on your discussion here.

*Hint: In the code, the time-step, `dt`, is used together with `dTdt` to compute `dT`.*

**/your discussion notes here/**

#### &#128721; **Stop here and check in with an instructor.**

&#9989; **Task 2.3:** Find a value of `dt` greater than 1 that can still be used to compute the temperature of the wire over time. Update your code from Task 1.3 using this new `dt` value. You should see the run time get even shorter than before.

In [13]:
# your answer here

This limitation with the time-step can be avoided. In this course, we use what is called an "explicit method" for FDMs, which runs the risk of divergence. To learn more about non-divergent FDMs, see these other numerical schemes:
- https://en.wikipedia.org/wiki/Finite_difference_method#Implicit_method
- https://en.wikipedia.org/wiki/Crank%E2%80%93Nicolson_method